# 02 — Jacobian Lens (JLens)

Fit $J_\ell = \mathbb{E}[\partial h_{\mathrm{final}}/\partial h_\ell]$, apply `unembed(h @ J.T)`, visualize layer readouts.

Docs: `docs/jlensreimplement/`

## 0. Setup

In [ ]:
# --- Colab / local bootstrap (Drive + wheels + swappable model) ---
# Drive layout expected:
#   MyDrive/multilingual-mechinterp/
#     dist/*.whl
#     data/all200questions_persianMiddleEastCulture.json
#     configs/  notebooks/  results/
#
# Edit MODEL_NAME in notebooks/colab_setup.py (Qwen2.5 now; Gemma later),
# or override below after bootstrap.

from pathlib import Path
import runpy

def _resolve_setup_script() -> Path:
    here = Path.cwd()
    candidates = [
        here / "colab_setup.py",
        here / "notebooks" / "colab_setup.py",
        here.parent / "notebooks" / "colab_setup.py",
        Path("/content/drive/MyDrive/multilingual-mechinterp/notebooks/colab_setup.py"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "colab_setup.py not found. Mount Drive with the project folder, "
        "or open the notebook from the repo."
    )

_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

# Session overrides (uncomment as needed):
# MODEL_NAME = "google/gemma-2-2b"
# MODEL_TRUST_REMOTE_CODE = False
# USE_TINY_OFFLINE = True   # demos without downloading HF weights

import matplotlib.pyplot as plt
import torch

from multilingual_mechinterp.utils import ensure_dir, load_config

cfg_path = CONFIG_DIR / "qwen25.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}
if "model" in cfg and not USE_TINY_OFFLINE:
    # keep notebook MODEL_NAME as source of truth; cfg is fallback metadata
    pass

print("Ready.")
print(" ROOT =", ROOT)
print(" DATA =", DATA_DIR)
print(" DIST =", DIST_DIR)
print(" MODEL =", MODEL_NAME, "| tiny=", USE_TINY_OFFLINE)

from multilingual_mechinterp.jlens import TinyDecoder, fit, run_jlens, JacobianLens, from_hf

OUT = ensure_dir(RESULTS_DIR / "jlens")


## Load experiment model

Uses `MODEL_NAME` from `colab_setup.py` (default **Qwen2.5**). Set `USE_TINY_OFFLINE=True` for demos without HF downloads.


In [ ]:
# Real model (Qwen now; change MODEL_NAME for Gemma later) OR tiny offline
# model = load_experiment_model()
# For gated Gemma: export HF_TOKEN=... or pass token=...

# Default path in analysis cells below uses tiny models for speed.
# Swap in `model = load_experiment_model()` when you are ready for Qwen/Gemma.
print("To load HF weights:", f"load_experiment_model({MODEL_NAME!r})")
print("Culture JSON:", culture_json_path(), "exists=", culture_json_path().exists())


## 1. Fit a lens on TinyDecoder (offline)

In [ ]:
model = TinyDecoder(n_layers=4, d_model=16, seed=0)
fit_prompts = [
    "The capital of France is Paris and it is lovely today",
    "Sparse autoencoders find interpretable features in LMs",
    "Jacobian lenses read what residuals are disposed to say",
    "Cross lingual culture questions about Hafez and Nowruz",
]
lens = fit(
    model,
    fit_prompts,
    source_layers=[0, 1, 2],
    target_layer=3,
    dim_batch=8,
    skip_first=0,
    max_seq_len=64,
)
ckpt = OUT / "tiny_lens.pt"
lens.save(str(ckpt))
print("n_prompts=", lens.n_prompts, "layers=", lens.source_layers, "->", ckpt)

## 2. Apply lens and plot top-token logits by layer

In [ ]:
prompt = "The capital of France is"
result = run_jlens(
    model, prompt, lens=lens, layers=[0, 1, 2],
    positions=[-1], top_k=8, use_jacobian=True,
)

# Compare Jacobian lens vs logit lens (identity transport)
logit = run_jlens(
    model, prompt, lens=lens, layers=[0, 1, 2],
    positions=[-1], top_k=8, use_jacobian=False,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, res, title in [
    (axes[0], result, "Jacobian lens"),
    (axes[1], logit, "Logit lens (no J)"),
]:
    for layer, toks in res.top_tokens.items():
        vals = [t["logit"] for t in toks[:5]]
        ax.plot(range(len(vals)), vals, marker="o", label=f"L{layer}")
    ax.set_title(title)
    ax.set_xlabel("rank")
    ax.set_ylabel("logit")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle(f"Readout for: {prompt!r}")
plt.tight_layout()
fig.savefig(OUT / "jlens_vs_logit.png", dpi=150)
plt.show()

for layer, toks in result.top_tokens.items():
    print(f"L{layer}:", [(t["token"], round(t["logit"], 3)) for t in toks[:4]])

## 3. Jacobian matrix heatmaps (optional diagnostic)

In [ ]:
fig, axes = plt.subplots(1, len(lens.source_layers), figsize=(12, 3.2))
if len(lens.source_layers) == 1:
    axes = [axes]
for ax, layer in zip(axes, lens.source_layers):
    J = lens.jacobians[layer].numpy()
    im = ax.imshow(J, cmap="coolwarm", vmin=-J.abs().max(), vmax=J.abs().max())
    ax.set_title(f"J layer {layer}")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
fig.savefig(OUT / "jacobian_heatmaps.png", dpi=150)
plt.show()

## 4. HF model (optional)

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from multilingual_mechinterp.jlens import from_hf, fit

tok = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")
hf = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-70m-deduped")
model = from_hf(hf, tok)
lens = fit(model, fit_prompts, checkpoint_path="results/jlens/fit.ckpt")
```
Use `skip_first=16` and longer prompts for paper-faithful fits.